# 03 - Gold : agrégats pour Claire, export PostgreSQL

La silver contient une ligne par vente. La gold calcule ce dont Claire a besoin pour décider : par département et par mois, le **nombre de ventes** (activité), le **prix au m² médian** (niveau de prix) et l'**écart à la médiane nationale**.

Trois tables, écrites en Delta dans le lac puis exportées vers PostgreSQL :
- `gold_departement` : une ligne par département et type de bien ;
- `gold_departement_mois` : une ligne par département, type de bien et mois ;
- `gold_national_mois` : la série nationale, pour le contexte.

`type_bien` vaut `Tous`, `Maison` ou `Appartement` : les prix diffèrent beaucoup entre maisons et appartements.

In [ ]:
import os
import sys
sys.path.append("/home/jovyan/work")

from pyspark.sql import functions as F

from outils.source import creer_session, creer_bucket, LAKE

ANNEE = 2025
SILVER = f"{LAKE}/silver/ventes_dvf_{ANNEE}"
GOLD = f"{LAKE}/gold"

PG_URL = os.environ["PG_URL"]
PG_USER = os.environ["PG_USER"]
PG_PASSWORD = os.environ["PG_PASSWORD"]
PILOTE_PG = "org.postgresql:postgresql:42.7.4"  

In [3]:
spark = creer_session("gold", paquets_supplementaires=[PILOTE_PG])
creer_bucket()

ventes = spark.read.format("delta").load(SILVER)
total = ventes.count()
print(f"Ventes en silver : {total:,}".replace(",", " "))

Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3. Le premier démarrage télécharge des jars : plusieurs minutes.
Session prête.
Bucket « dvf » déjà présent.
Ventes en silver : 804 803


## 1.combien de ventes par département ?
Une médiane calculée sur quelques dizaines de ventes est peu fiable. On regarde la distribution du nombre de ventes par département pour fixer un seuil de fiabilité.

In [ ]:
par_dep = ventes.groupBy("code_departement").count()
print("Départements :", par_dep.count())

q = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]
for p, v in zip(q, par_dep.approxQuantile("count", q, 0.0)):
    print(f"quantile {p:>4} : {v:>8,.0f} ventes par département".replace(",", " "))

par_dep.orderBy("count").show(10)

niveaux = {
    "département × type": ventes.groupBy("code_departement", "type_local").count(),
    "département × mois": ventes.groupBy("code_departement", "annee", "mois").count(),
    "département × type × mois": ventes.groupBy("code_departement", "type_local", "annee", "mois").count(),
}
for nom, t in niveaux.items():
    n_cases = t.count()
    print(f"{nom} : {n_cases} cases")
    for s in [10, 20, 30, 50, 100]:
        n = t.filter(F.col("count") < s).count()
        print(f"   moins de {s:>3} ventes : {n:>5} cases ({n / n_cases:.1%})")

Départements : 97
quantile  0.0 :      741 ventes par département
quantile  0.1 :    1 895 ventes par département
quantile 0.25 :    3 267 ventes par département
quantile  0.5 :    6 640 ventes par département
quantile 0.75 :   11 197 ventes par département
quantile  1.0 :   29 894 ventes par département
+----------------+-----+
|code_departement|count|
+----------------+-----+
|              48|  741|
|             973|  755|
|              23| 1380|
|              52| 1536|
|             972| 1586|
|              90| 1592|
|             971| 1667|
|              15| 1723|
|              55| 1821|
|              2A| 1895|
+----------------+-----+
only showing top 10 rows
département × type : 194 cases
   moins de  10 ventes :     0 cases (0.0%)
   moins de  20 ventes :     0 cases (0.0%)
   moins de  30 ventes :     0 cases (0.0%)
   moins de  50 ventes :     0 cases (0.0%)
   moins de 100 ventes :     1 cases (0.5%)
département × mois : 1164 cases
   moins de  10 ventes :     0 cases

In [5]:
SEUIL_VENTES =  741
assert SEUIL_VENTES is not None, "Fixer SEUIL_VENTES d'après la distribution ci-dessus."

## 2. Préparer les trois niveaux de `type_bien`
Chaque vente apparaît deux fois : une avec `Tous`, une avec son type réel (Maison ou Appartement). Les médianes ne s'additionnent pas : on les recalcule pour chaque niveau.

In [6]:
tous = ventes.withColumn("type_bien", F.lit("Tous"))
detail = ventes.withColumn("type_bien", F.col("type_local"))
base = tous.unionByName(detail)

national = base.groupBy("type_bien").agg(F.median("prix_m2").alias("prix_m2_median_national"))

## 3. Les trois tables gold

In [ ]:
# Par département
dep = (base.groupBy("code_departement", "type_bien")
       .agg(F.count("*").alias("nb_ventes"), F.median("prix_m2").alias("prix_m2_median"),
            F.round(F.median("valeur_fonciere"), 0).alias("prix_median"),
            F.round(F.median("surface_bati"), 0).alias("surface_mediane"))
       .join(national, "type_bien")
       .withColumn("ecart_national", F.col("prix_m2_median") / F.col("prix_m2_median_national") - 1)
       .withColumn("fiable", F.col("nb_ventes") >= SEUIL_VENTES)
       .withColumn("prix_m2_median", F.round("prix_m2_median", 0))
       .withColumn("prix_m2_median_national", F.round("prix_m2_median_national", 0))
       .withColumn("ecart_national", F.round("ecart_national", 3)))

# Par département et par mois (le drapeau « fiable » est évalué sur le nombre de ventes de chaque case)
mois = (base.groupBy("code_departement", "type_bien", "annee", "mois")
        .agg(F.count("*").alias("nb_ventes"), F.round(F.median("prix_m2"), 0).alias("prix_m2_median"),
             F.round(F.median("valeur_fonciere"), 0).alias("prix_median"),
             F.round(F.median("surface_bati"), 0).alias("surface_mediane"))
        .withColumn("fiable", F.col("nb_ventes") >= SEUIL_VENTES))

# Série nationale par mois
nat_mois = (base.groupBy("type_bien", "annee", "mois")
            .agg(F.count("*").alias("nb_ventes"), F.round(F.median("prix_m2"), 0).alias("prix_m2_median")))

## 4. Contrôles
Pour chaque table, le total des ventes du niveau `Tous` doit être égal au nombre de ventes de la silver : sinon, des ventes ont été perdues ou comptées deux fois.

In [8]:
def somme_tous(t):
    return t.filter(F.col("type_bien") == "Tous").agg(F.sum("nb_ventes")).first()[0]

for nom, t in [("gold_departement", dep), ("gold_departement_mois", mois), ("gold_national_mois", nat_mois)]:
    s = somme_tous(t)
    print(f"{nom:<24} total des ventes : {s:,}".replace(",", " "), "| OK" if s == total else f"| ERREUR (silver : {total})")
    assert s == total, f"{nom} : le total ne correspond pas à la silver."

print("Départements fiables :", dep.filter((F.col("type_bien") == "Tous") & F.col("fiable")).count(),
      "sur", dep.filter(F.col("type_bien") == "Tous").count())
(dep.filter((F.col("type_bien") == "Tous") & F.col("fiable"))
    .orderBy(F.desc("nb_ventes")).select("code_departement", "nb_ventes", "prix_m2_median", "ecart_national").show(10))

gold_departement         total des ventes : 804 803 | OK
gold_departement_mois    total des ventes : 804 803 | OK
gold_national_mois       total des ventes : 804 803 | OK
Départements fiables : 97 sur 97
+----------------+---------+--------------+--------------+
|code_departement|nb_ventes|prix_m2_median|ecart_national|
+----------------+---------+--------------+--------------+
|              59|    29894|        2113.0|        -0.193|
|              75|    29229|        9726.0|         2.717|
|              13|    25566|        3606.0|         0.378|
|              06|    23771|        4926.0|         0.883|
|              33|    22128|        3387.0|         0.295|
|              83|    21727|        3978.0|         0.521|
|              69|    21351|        3687.0|         0.409|
|              34|    20206|        3263.0|         0.247|
|              44|    19870|        3058.0|         0.169|
|              31|    19145|        2840.0|         0.086|
+----------------+---------+-

## 5. Écriture en Delta dans le lac

In [10]:
TABLES = {"gold_departement": dep, "gold_departement_mois": mois, "gold_national_mois": nat_mois}

for nom, t in TABLES.items():
    t.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{GOLD}/{nom}")
    print("écrit :", f"{GOLD}/{nom}")

écrit : s3a://dvf/gold/gold_departement
écrit : s3a://dvf/gold/gold_departement_mois
écrit : s3a://dvf/gold/gold_national_mois


## 6. Export vers PostgreSQL (JDBC)


In [11]:
def options_pg(table):
    return {"url": PG_URL, "dbtable": table, "user": PG_USER, "password": PG_PASSWORD, "driver": "org.postgresql.Driver"}

for nom in TABLES:
    t = spark.read.format("delta").load(f"{GOLD}/{nom}")
    t.write.format("jdbc").options(**options_pg(nom)).mode("overwrite").save()
    n_pg = spark.read.format("jdbc").options(**options_pg(nom)).load().count()
    print(f"{nom:<24} lac : {t.count():>6} lignes | PostgreSQL : {n_pg:>6} lignes")
    assert t.count() == n_pg, f"{nom} : le nombre de lignes diffère entre le lac et PostgreSQL."

gold_departement         lac :    291 lignes | PostgreSQL :    291 lignes
gold_departement_mois    lac :   3492 lignes | PostgreSQL :   3492 lignes
gold_national_mois       lac :     36 lignes | PostgreSQL :     36 lignes


In [12]:
spark.stop()